## Importing libraries

In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
import spacy 
nlp = spacy.load("en_core_web_sm")

### Read reviews data

In [3]:
# 1. Download the file from the URL to the Colab disk
!wget "https://raw.githubusercontent.com/aqwertyuiop48/upgrad_programming/refs/heads/main/2_Course_continuation/_3_NLP/_2_Syntactic_processing/_1_Intro_to_Syntactic_processing_and_POS_tagging/_2_POS_Tagging_Case_Study_(Google_colab)/Dataset/Samsung.txt"

# 2. Open the local file (matching your original code)
con = open("Samsung.txt", 'r', encoding="utf-8")
samsung_reviews = con.read()
con.close()

--2026-02-25 06:00:13--  https://raw.githubusercontent.com/aqwertyuiop48/upgrad_programming/refs/heads/main/2_Course_continuation/_3_NLP/_2_Syntactic_processing/_1_Intro_to_Syntactic_processing_and_POS_tagging/_2_POS_Tagging_Case_Study_(Google_colab)/Dataset/Samsung.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7491391 (7.1M) [text/plain]
Saving to: ‘Samsung.txt.2’

Samsung.txt.2       100%[===================>]   7.14M  --.-KB/s    in 0.05s   

2026-02-25 06:00:13 (133 MB/s) - ‘Samsung.txt.2’ saved [7491391/7491391]



In [4]:
len(samsung_reviews.split("\n"))

46355

### Dataset is a text file where each review is in a new line

In [5]:
samsung_reviews.split("\n")[0:4]

["I feel so LUCKY to have found this used (phone to us & not used hard at all), phone on line from someone who upgraded and sold this one. My Son liked his old one that finally fell apart after 2.5+ years and didn't want an upgrade!! Thank you Seller, we really appreciate it & your honesty re: said used phone.I recommend this seller very highly & would but from them again!!",
 'nice phone, nice up grade from my pantach revue. Very clean set up and easy set up. never had an android phone but they are fantastic to say the least. perfect size for surfing and social media. great phone samsung',
 'Very pleased',
 'It works good but it goes slow sometimes but its a very good phone I love it']

### Will our hypothesis hold on real world data? `Product features---POS_NOUN`

In [6]:
review1=samsung_reviews.split("\n")[0]
review1=nlp(review1)

### Lets do nlp parse on part of one review in our dataset

In [7]:
for tok in review1[0:10]:
    print(tok.text,"---",tok.lemma_,"---",tok.pos_)

I --- I --- PRON
feel --- feel --- VERB
so --- so --- ADV
LUCKY --- lucky --- ADJ
to --- to --- PART
have --- have --- AUX
found --- find --- VERB
this --- this --- PRON
used --- used --- ADJ
( --- ( --- PUNCT


#### Real world data is usually messy, observe the words `found` and `used`

In [8]:
pos = []
lemma = []
text = []
for tok in review1:
    pos.append(tok.pos_)
    lemma.append(tok.lemma_)
    text.append(tok.text)

In [9]:
nlp_table = pd.DataFrame({'text':text,'lemma':lemma,'pos':pos})
nlp_table.head()

,text,lemma,pos
0,I,I,PRON
1,feel,feel,VERB
2,so,so,ADV
3,LUCKY,lucky,ADJ
4,to,to,PART


In [10]:
## Get most frequent lemma forms of nouns
nlp_table[nlp_table['pos']=='NOUN']['lemma'].value_counts()

,count
lemma,
phone,3
one,2
line,1
year,1
upgrade,1
honesty,1
re,1
seller,1


#### It seems possible that if we extract all the nouns from the reviews and look at the top 5 most frequent lemmatised noun forms, we will be able to identify `What people are talking about?`

### Lets repeat this experiment on a larger set of reviews

In [11]:
nouns = []
for review in samsung_reviews.split("\n")[0:100]:
    doc = nlp(review)
    for tok in doc:
        if tok.pos_=="NOUN":
            nouns.append(tok.lemma_.lower())

### Lets add some way of keeping track of time

In [12]:
from tqdm import tqdm
nouns = []
for review in tqdm(samsung_reviews.split("\n")[0:1000]):
    doc = nlp(review)
    for tok in doc:
        if tok.pos_=="NOUN":
            nouns.append(tok.lemma_.lower())
pd.Series(nouns).value_counts().head(5)

100%|██████████| 1000/1000 [00:12<00:00, 82.99it/s]


,count
phone,1214
time,92
battery,89
price,87
screen,86


In [13]:
len(samsung_reviews.split("\n"))

46355

### Did you notice anything? What do you think will be the time taken to process all the reviews?

In [14]:
(46355//1000)*17

782

In [15]:
782//60

13

## Summary
- POS tag based rule seems to be working well
- We need to figure out a way to reduce the time taken to process reviews

In [16]:
import datetime, pytz; 
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))

Current Time in IST: 2026-02-25 11:30:30


In [ ]:
# --- gcolab wrapper: list files in outputs/ ---
import os, datetime
_out_dir = 'outputs'
os.makedirs(_out_dir, exist_ok=True)
print(f"Files in {_out_dir}:")
_files = []
for _fn in os.listdir(_out_dir):
    _fp = os.path.join(_out_dir, _fn)
    if os.path.isfile(_fp):
        _st = os.stat(_fp)
        _files.append((datetime.datetime.fromtimestamp(_st.st_mtime).strftime('%Y-%m-%d %H:%M:%S'), _st.st_size, _fn))
_files.sort()
for _m, _s, _n in _files:
    if _s < 1024:
        _h = f"{_s} B"
    elif _s < 1024*1024:
        _h = f"{_s/1024:.2f} KB"
    else:
        _h = f"{_s/(1024*1024):.2f} MB"
    print(f"{_m}  {_h:>10}  {_n}")


In [ ]:
# --- gcolab wrapper: zip outputs/ -> outputs.zip ---
import os, shutil
os.makedirs('outputs', exist_ok=True)
shutil.make_archive('outputs', 'zip', 'outputs')
print('Created outputs.zip')
